# Triple Precision Evaluation

Manual + LLM-judge verification of 100 random SPO triples extracted by Mistral 7B from Open Food Facts product descriptions. Per proposal §V.C, a triple is correct if both entities and the predicate accurately reflect the source text.

In [ ]:
import sys, json, random
from pathlib import Path
sys.path.insert(0, '/scratch/smehta90/Clickless AI')
import pandas as pd
from src.knowledge_graph.spo_extractor import load_triples
from src.llm import ollama_client as llm

random.seed(42)
triples = load_triples()
print(f'total triples: {len(triples)}')
sample = random.sample(triples, k=min(100, len(triples)))
df = pd.DataFrame(sample)
df.head()

In [ ]:
JUDGE_PROMPT = '''You are a fact-checking assistant for a food knowledge graph.
Decide whether the following Subject-Predicate-Object triple is a correct
factual statement that could plausibly be derived from the source product.

Source product: {product}
Triple: ({s}) -[{p}]-> ({o})

Respond with ONLY a JSON object {{"correct": true|false, "reason": "..."}}.'''

def judge(row):
    prompt = JUDGE_PROMPT.format(product=row.get('product',''), s=row['s'], p=row['p'], o=row['o'])
    try:
        result = llm.generate_json(prompt, role='general')
        return bool(result.get('correct', False))
    except Exception:
        return None

judgements = [judge(r) for _, r in df.iterrows()]
df['correct'] = judgements
valid = df.dropna(subset=['correct'])
precision = valid['correct'].mean() if len(valid) else 0.0
print(f'judged: {len(valid)} | triple precision: {precision:.3f}')
df.to_csv('/scratch/smehta90/Clickless AI/evaluation/results/triple_precision_sample.csv', index=False)
Path('/scratch/smehta90/Clickless AI/evaluation/results/triple_precision.json').write_text(json.dumps({'n': len(valid), 'precision': precision}, indent=2))

**Result**: precision is reported in `evaluation/results/triple_precision.json`. Target ≥0.85.